In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

In [ ]:
# assuming that line gap t = 1
def Buffon_simulation(l, drop_interval, total_drops, crosses):
    
    for _ in range(drop_interval):
        x1 = random.uniform(0, 1)
        theta = random.uniform(-np.pi, np.pi)
        x2 = x1 + l * np.cos(theta)
        
        crosses += abs(int(np.floor(x2)) - int(np.floor(x1))) 
    
    if crosses == 0:
        pi_estimate = float("nan")
    else:
        pi_estimate = 2 * l * total_drops / crosses

    return crosses, pi_estimate

In [ ]:
# Buffon's needle l/t pi convergence comparison
min_ratio = 0.01
max_ratio = 1000
num_ratios = 10

# taking line gap t = 1, hence l = ratios
ratio_list = np.linspace(min_ratio, max_ratio, num_ratios)
cross_list = [0] * num_ratios

# plot initialisation
fig, ax = plt.subplots(nrows=2, figsize=(8, 10), dpi=80)
fig.suptitle(
    r"Buffon's Needle $\pi$ Convergence" '\n'
    r"Comparison for $\ell/t$ Ratio", 
    fontsize=18, 
    fontweight="bold"
)
plt.subplots_adjust(left=0.1, right=0.9, top=0.88, bottom=0.05, hspace=0.3)

bar_width = (max_ratio - min_ratio) / num_ratios
bar_list = ax[0].bar(ratio_list, [0]*num_ratios, width=1.05*bar_width, zorder=3)

colours = plt.cm.plasma(np.linspace(1, 0, num_ratios))
for i in range(num_ratios):
        bar_list[i].set_facecolor(colours[i])

diffs = []
for i in range(num_ratios):
    line, = ax[1].plot([], [], color=colours[i], linewidth=1.5)
    diffs.append(line,)

for i in [0, 1]:
    ax[i].grid(True, which='major', linestyle='-', linewidth=0.8)
    ax[i].grid(True, which='minor', linestyle=':', linewidth=0.5)
    ax[i].minorticks_on()

# ax[0] plot formatting
ax[0].set_ylim(0, 6)

ax[0].axhline(np.pi, linestyle="--", color="red", linewidth=1.5, zorder=2)
ax_pi = ax[0].twinx()
ax_pi.set_ylim(ax[0].get_ylim())
ax_pi.set_yticks([np.pi])
ax_pi.set_yticklabels([r'$\pi$'], color="red")
ax_pi.tick_params(axis='y', length=0)

bar_title = ax[0].set_title("Needles Dropped", fontsize=15, x=1, ha="right")
ax[0].set_xlabel(r"Needle length to line gap ratio $\ell/t$", fontsize=12)
ax[0].set_ylabel(r"$\pi$ estimate", fontsize=12)

# ax[1] plot formatting
ax[1].set_yscale("log")

ax[1].set_title(r"Difference Between $\pi$ Estimate and Actual", fontsize=15)
ax[1].set_xlabel(r"Needle dropped", fontsize=12)
ax[1].set_ylabel(r"$|\pi_{\text{est}} - \pi|$", fontsize=12)
ax[1].yaxis.set_label_coords(-0.08, 0.5)

# run Buffon's needle simulation
drop_interval = 10
rounds = 100

# make the animation end on (rounds * drop_interval) needles dropped
rounds += 1

drops = []
diff_list = [[0] * (rounds - 1) for _ in range(num_ratios)]

def update(frame):

    total_drops = frame * drop_interval
    bar_title.set_text(f"{total_drops} Needles Dropped")

    if frame > 0:

        drops.append(total_drops)
        
        for i in range(num_ratios):

            # find number of crosses and current pi estimate for ratio
            cross_list[i], pi_est = Buffon_simulation(
                ratio_list[i], drop_interval, total_drops, cross_list[i]
            )
            
            bar_list[i].set_height(pi_est)

            # find difference between current pi estimate and actual value
            diff_list[i][frame-1] = abs(np.pi - pi_est)
            diffs[i].set_data(drops, diff_list[i][:frame])

            if frame == 1:
                ax[1].scatter(
                    drop_interval, diff_list[i][frame-1], 
                    s=1.5, color=colours[i], zorder=10
                )

        ax[1].relim()
        ax[1].autoscale_view()

anim = FuncAnimation(fig, update, frames=rounds, blit=False)
anim.save(
    f"bn_conv_{min_ratio}-{max_ratio}_{(rounds-1)*drop_interval}_drops.gif", 
    writer=PillowWriter(fps=(rounds+1)/10)
)
plt.close(fig)
# HTML(anim.to_jshtml())